In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install duckdb==1.4.4 polars==1.40.0 --quiet

In [ ]:
# ── Cell 2: Mount Drive and copy DB to local (Colab only) ─────────────────
from pathlib import Path
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DB = Path("/content/drive/MyDrive/b3_data/db/b3_data.duckdb")
    LOCAL_DB = Path("/content/b3_data.duckdb")

    if not DRIVE_DB.exists():
        raise FileNotFoundError(
            f"Database not found at {DRIVE_DB}.\n"
            "Run colab_ingest.ipynb first to build the database."
        )

    print(f"Copying DB from Drive → {LOCAL_DB} ...")
    shutil.copy2(DRIVE_DB, LOCAL_DB)
    size_mb = LOCAL_DB.stat().st_size / 1_048_576
    print(f"Ready. DB size: {size_mb:.1f} MB")
    DB_PATH = LOCAL_DB
else:
    DB_PATH = Path("/data/b3.duckdb")
    print(f"Using local DB: {DB_PATH}")

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import duckdb
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 4: Parameters — edit this cell to run a new analysis ─────────────
# ─────────────────────────────────────────────────────────────────────────
TICKER      = "PETR4"      # underlying stock ticker
START_DATE  = "2024-01-02" # analysis start (must be a trading day with data)
TARGET_DATE = "2024-06-30" # analysis end; option expiry is nearest to this
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Cell 5: Open connection ───────────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))
con.execute("SET threads TO 2")
con.execute("SET memory_limit = '4GB'")
print("Connection open. DuckDB version:", duckdb.__version__)

In [ ]:
# ── Cell 6: Select ATM options nearest to target expiry ───────────────────
# Strategy:
#   1. Get stock closing price and especi on START_DATE → sets target strike
#      and share class (PN/ON) used to filter matching options.
#   2. Option tickers use a 4-char root (e.g. "PETR" for PETR3/PETR4);
#      especi disambiguates which underlying class the option belongs to.
#   3. Find the expiry date nearest to TARGET_DATE with both a CALL and PUT.
#   4. For that expiry, pick the CALL and PUT with strike closest to
#      the stock price on START_DATE (nearest ATM).

df_selection = con.sql(f"""
    WITH stock_start AS (
        SELECT preult AS start_price, especi
        FROM cotacoes
        WHERE codneg = '{TICKER}'
          AND datpre = '{START_DATE}'::DATE
          AND codbdi = '02'
    ),
    options_on_start AS (
        SELECT
            codneg,
            tpmerc,
            preexe,
            datven,
            ABS(preexe - (SELECT start_price FROM stock_start)) AS strike_diff,
            ABS(DATEDIFF('day', datven, '{TARGET_DATE}'::DATE))  AS expiry_diff
        FROM cotacoes
        WHERE datpre = '{START_DATE}'::DATE
          AND tpmerc IN ('070', '080')
          AND codneg LIKE '{TICKER[:4]}%'
          AND SUBSTR(especi, 1, 2) = (SELECT SUBSTR(especi, 1, 2) FROM stock_start)
          AND NOT regexp_matches(codneg, 'W[1-5]$')
          AND datven > '{START_DATE}'::DATE
          AND datven <= '{TARGET_DATE}'::DATE
    ),
    best_expiry AS (
        SELECT datven
        FROM options_on_start
        GROUP BY datven
        HAVING
            COUNT(CASE WHEN tpmerc = '070' THEN 1 END) > 0
            AND COUNT(CASE WHEN tpmerc = '080' THEN 1 END) > 0
        ORDER BY MIN(expiry_diff)
        LIMIT 1
    ),
    best_call AS (
        SELECT codneg, preexe, datven
        FROM options_on_start
        WHERE tpmerc = '070'
          AND datven = (SELECT datven FROM best_expiry)
        ORDER BY strike_diff
        LIMIT 1
    ),
    best_put AS (
        SELECT codneg, preexe, datven
        FROM options_on_start
        WHERE tpmerc = '080'
          AND datven = (SELECT datven FROM best_expiry)
        ORDER BY strike_diff
        LIMIT 1
    )
    SELECT
        (SELECT start_price FROM stock_start) AS stock_start_price,
        (SELECT especi     FROM stock_start)  AS stock_especi,
        (SELECT codneg FROM best_call)        AS call_ticker,
        (SELECT preexe FROM best_call)        AS call_strike,
        (SELECT codneg FROM best_put)         AS put_ticker,
        (SELECT preexe FROM best_put)         AS put_strike,
        (SELECT datven FROM best_expiry)      AS expiry_date
""").df()

# Validate: stock data must exist on START_DATE
stock_check = con.sql(f"""
    SELECT preult FROM cotacoes
    WHERE codneg = '{TICKER}' AND datpre = '{START_DATE}'::DATE AND codbdi = '02'
""").df()

if len(stock_check) == 0:
    raise ValueError(
        f"No quote found for {TICKER} on {START_DATE}.\n"
        "Check the ticker and ensure the date is a trading day with ingested data."
    )

if df_selection["call_ticker"].isna().all():
    raise ValueError(
        f"No options found for {TICKER} between {START_DATE} and {TARGET_DATE}.\n"
        "Try adjusting the date range or verify the ticker has options data."
    )

row          = df_selection.iloc[0]
START_PRICE  = row["stock_start_price"]
STOCK_ESPECI = row["stock_especi"].strip()
CALL_TICKER  = row["call_ticker"]
CALL_STRIKE  = row["call_strike"]
PUT_TICKER   = row["put_ticker"]
PUT_STRIKE   = row["put_strike"]
EXPIRY_DATE  = str(row["expiry_date"])[:10]

print(f"Stock:  {TICKER:<12} ({STOCK_ESPECI}) — start price R$ {START_PRICE:.2f}")
print(f"CALL:   {CALL_TICKER:<12} — strike R$ {CALL_STRIKE:.2f} — expiry {EXPIRY_DATE}")
print(f"PUT:    {PUT_TICKER:<12} — strike R$ {PUT_STRIKE:.2f} — expiry {EXPIRY_DATE}")

In [ ]:
# ── Cell 7: Fetch price series ────────────────────────────────────────────
df_stock = con.sql(f"""
    SELECT datpre, preult AS price
    FROM cotacoes
    WHERE codneg = '{TICKER}'
      AND codbdi = '02'
      AND datpre BETWEEN '{START_DATE}'::DATE AND '{TARGET_DATE}'::DATE
    ORDER BY datpre
""").df()

df_call = con.sql(f"""
    SELECT datpre, preult AS price
    FROM cotacoes
    WHERE codneg = '{CALL_TICKER}'
      AND tpmerc = '070'
      AND datpre BETWEEN '{START_DATE}'::DATE AND '{EXPIRY_DATE}'::DATE
    ORDER BY datpre
""").df()

df_put = con.sql(f"""
    SELECT datpre, preult AS price
    FROM cotacoes
    WHERE codneg = '{PUT_TICKER}'
      AND tpmerc = '080'
      AND datpre BETWEEN '{START_DATE}'::DATE AND '{EXPIRY_DATE}'::DATE
    ORDER BY datpre
""").df()

print(f"Stock rows: {len(df_stock)} | CALL rows: {len(df_call)} | PUT rows: {len(df_put)}")

In [ ]:
# ── Cell 8: Dual Y-axis performance chart ─────────────────────────────────
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Stock price — left axis
fig.add_trace(
    go.Scatter(
        x=df_stock["datpre"], y=df_stock["price"],
        name=f"{TICKER} (Stock)",
        line=dict(color="white", width=2),
    ),
    secondary_y=False,
)

# CALL price — right axis
fig.add_trace(
    go.Scatter(
        x=df_call["datpre"], y=df_call["price"],
        name=f"{CALL_TICKER} (CALL · R$ {CALL_STRIKE:.2f} · exp {EXPIRY_DATE})",
        line=dict(color="#00cc44", width=2),
    ),
    secondary_y=True,
)

# PUT price — right axis
fig.add_trace(
    go.Scatter(
        x=df_put["datpre"], y=df_put["price"],
        name=f"{PUT_TICKER} (PUT · R$ {PUT_STRIKE:.2f} · exp {EXPIRY_DATE})",
        line=dict(color="#ff4466", width=2),
    ),
    secondary_y=True,
)

# Vertical dotted markers at START_DATE and TARGET_DATE
# add_vline requires a Unix timestamp in milliseconds on date axes
for date_str, label in [(START_DATE, "Start"), (EXPIRY_DATE, "Expiry"), (TARGET_DATE, "Target")]:
    x_ms = pd.Timestamp(date_str).timestamp() * 1000
    fig.add_vline(
        x=x_ms,
        line=dict(color="rgba(200,200,200,0.4)", dash="dot", width=1.5),
        annotation_text=label,
        annotation_font_color="rgba(200,200,200,0.8)",
        annotation_position="top left",
    )

fig.update_layout(
    title=dict(
        text=(
            f"{TICKER} — Option Performance ({START_DATE} → {TARGET_DATE})<br>"
            f"<sup>Expiry: {EXPIRY_DATE} "
            f"| CALL strike: R$ {CALL_STRIKE:.2f} "
            f"| PUT strike: R$ {PUT_STRIKE:.2f}</sup>"
        ),
        font_color="white",
    ),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0d0d1a",
    font_color="white",
    legend=dict(
        bgcolor="rgba(255,255,255,0.05)",
        bordercolor="rgba(255,255,255,0.2)",
        borderwidth=1,
    ),
    hovermode="x unified",
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.08)", zeroline=False)
fig.update_yaxes(
    title_text=f"{TICKER} Price (R$)",
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    secondary_y=False,
)
fig.update_yaxes(
    title_text="Option Price (R$)",
    showgrid=False,
    secondary_y=True,
)

display(fig)

In [ ]:
# ── Cell 9: Close connection and sync DB back to Drive ────────────────────
con.close()

if IN_COLAB:
    shutil.copy2(LOCAL_DB, DRIVE_DB)
    print(f"DB synced to Drive. Size: {DRIVE_DB.stat().st_size / 1_048_576:.1f} MB")
else:
    print("Connection closed.")